# Example 05 — Data Processing: from DICOM slices to files three.js can load

The browser cannot read `.dcm` files, and running marching cubes on 4 million voxels in JavaScript is slow.
So we do the heavy work **once, in Python**, and save the results as simple files that the web pages load.

```
68 .dcm files ──► sort by z ──► 3D array in HU ──┬──► ct_volume.bin + .json     (index_01, 02, 03)
 (Week 11 data)                                   │
                                                  └──► smooth ──► marching cubes ──► surfaces/*.json  (index_04)
```

| Step | What we do | Output |
|---|---|---|
| 1 | Read the DICOM files and sort them by height | 68 sorted slices |
| 2 | Stack them into one volume in Hounsfield Units | `volume[z, y, x]` |
| 3 | Export the volume for the slice and filter pages | `data/ct_volume.bin`, `data/ct_volume.json` |
| 4 | Make a smaller, smoother copy for marching cubes | `small[z, y, x]` |
| 5 | Extract iso-surfaces with marching cubes | vertices + triangles |
| 6 | Check one surface with matplotlib | a 3D plot |
| 7 | Export the surfaces for three.js | `data/surfaces/*.json` |

Steps 1 and 2 are explained in detail in the **Week 11** notebook (`Week-11/Medical-Data-DICOM/dicom_processor.ipynb`).
Here we only repeat the code.

**Setup**

```sh
pip install pydicom numpy scipy scikit-image matplotlib
```

In [ ]:
import glob
import json
import os

import numpy as np
import pydicom
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.ndimage import gaussian_filter
from skimage import measure

# The Week 11 CT scan, and the folder where the web pages look for data
DICOM_FOLDER = '../../../Week-11/Medical-Data-DICOM/DICOM_data'
OUT_FOLDER = '../data'

## Step 1: Read the DICOM files and sort them by height

File names are only labels. The real position of a slice is `ImagePositionPatient[2]` (its height in mm),
so we sort by that. Slice 0 is then the **bottom** of the scan (upper abdomen) and the last slice is the **top** (shoulders).

In [ ]:
files = sorted(glob.glob(os.path.join(DICOM_FOLDER, '*.dcm')))
assert files, f'No DICOM files found in {DICOM_FOLDER}'

slices = [pydicom.dcmread(f) for f in files]
assert len({ds.SeriesInstanceUID for ds in slices}) == 1, 'Files come from different scans'

slices.sort(key=lambda ds: float(ds.ImagePositionPatient[2]))   # bottom -> top
print(f'{len(slices)} slices, from z = {slices[0].ImagePositionPatient[2]} mm to z = {slices[-1].ImagePositionPatient[2]} mm')

## Step 2: Build the 3D volume in Hounsfield Units (HU)

```
HU = raw_value × RescaleSlope + RescaleIntercept
```

| Tissue | Approximate HU |
|---|---|
| Air | −1000 |
| Lung | −900 to −500 |
| Fat | −150 to −50 |
| Water | 0 |
| Soft tissue | +20 to +80 |
| Bone | +300 and more |

The array axes are `[z, y, x]`: slice (bottom → top), row (front → back of the body), column (across the body).
One voxel is **not a cube**: it is 0.736 × 0.736 mm across but 5 mm tall.

In [ ]:
volume = np.stack([
    ds.pixel_array * float(ds.RescaleSlope) + float(ds.RescaleIntercept)
    for ds in slices
]).astype(np.float32)

z_positions = [float(ds.ImagePositionPatient[2]) for ds in slices]
slice_spacing = float(np.abs(np.diff(z_positions)).mean())
row_spacing, col_spacing = (float(v) for v in slices[0].PixelSpacing)
spacing = (slice_spacing, row_spacing, col_spacing)      # mm, in the order (z, y, x)

print('Volume shape (z, y, x):', volume.shape)
print('Voxel size in mm (z, y, x):', spacing)
print('HU range:', volume.min(), 'to', volume.max())

## Step 3: Export the volume for the slice and filter pages (index_01 – index_03)

512 × 512 × 68 voxels is 17.8 million numbers, too much for a simple web page.
We **downsample** each slice by 2: every 2 × 2 block of pixels is averaged into one pixel, so a slice becomes 256 × 256.
We keep every slice in z, because the slices are already thick (5 mm).

```
  2 × 2 block            one pixel
  ┌────┬────┐
  │ 10 │ 20 │   mean
  ├────┼────┤  ──────►   ┌────┐
  │ 30 │ 40 │            │ 25 │
  └────┴────┘            └────┘
```

`reshape(z, 256, 2, 256, 2)` groups each 2 × 2 block into its own two axes, and `.mean(axis=(2, 4))` averages them.

We save the values as **binary** `int16` (2 bytes per voxel, 8.9 MB) instead of JSON text, which would be about 3 times larger.
JavaScript reads the file straight into an `Int16Array`.
A small JSON file stores the size and voxel spacing, which the page needs to read the binary file.

> **Privacy:** only the pixel values and the voxel size are exported. No header fields (patient information) are copied.

In [ ]:
def downsample(vol, factor):
    # Average every factor x factor block of pixels inside each slice (z is not changed)
    z, y, x = vol.shape
    return vol.reshape(z, y // factor, factor, x // factor, factor).mean(axis=(2, 4))

web_volume = downsample(volume, 2)
web_spacing = (slice_spacing, row_spacing * 2, col_spacing * 2)

# int16 is enough for HU (-1024 ... 3071). '<i2' = little-endian 2-byte integer, the byte order browsers use.
web_volume = np.clip(np.round(web_volume), -1024, 3071).astype('<i2')

os.makedirs(OUT_FOLDER, exist_ok=True)
web_volume.tofile(os.path.join(OUT_FOLDER, 'ct_volume.bin'))

meta = {
    'file': 'ct_volume.bin',
    'type': 'int16 little-endian, Hounsfield Units',
    'order': '[z][y][x]: index = z * rows * columns + y * columns + x',
    'slices': int(web_volume.shape[0]),
    'rows': int(web_volume.shape[1]),
    'columns': int(web_volume.shape[2]),
    'spacing_mm': {'z': round(web_spacing[0], 3), 'y': round(web_spacing[1], 3), 'x': round(web_spacing[2], 3)},
    'min_hu': int(web_volume.min()),
    'max_hu': int(web_volume.max()),
}
with open(os.path.join(OUT_FOLDER, 'ct_volume.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))
print(f"ct_volume.bin: {os.path.getsize(os.path.join(OUT_FOLDER, 'ct_volume.bin')) / 1e6:.1f} MB")

## Step 4: Prepare a smaller, smoother volume for marching cubes

Marching cubes places triangles wherever the data crosses the iso-value. Two problems with the raw data:

1. **Size:** on the full-resolution volume the skin surface has hundreds of thousands of triangles. That is too many for a web page.
   We downsample each slice by 4 (512 → 128 pixels, about 2.9 mm per pixel).
2. **Noise:** CT values are noisy, and every noisy voxel near the iso-value creates tiny extra triangles.
   A **Gaussian filter** replaces each voxel by a weighted average of its neighbours, which removes most of that noise.

`sigma` is how far the averaging reaches, in voxels. A small value (0.5) removes noise but keeps thin bones such as the ribs.
Try `sigma = 1` or `2`: the surfaces get smoother, but the dense-bone surface at 700 HU disappears, because the blur
mixes the thin bones with the soft tissue around them and no voxel stays above 700 HU.

In [ ]:
small = downsample(volume, 4)
small_spacing = (slice_spacing, row_spacing * 4, col_spacing * 4)

sigma = 0.5
small = gaussian_filter(small, sigma=sigma)

print('Volume for marching cubes (z, y, x):', small.shape)
print('Voxel size in mm (z, y, x):', tuple(round(s, 3) for s in small_spacing))

## Step 5: Extract iso-surfaces with marching cubes

An **iso-surface** is the 3D version of a contour line: the surface where the data crosses one value.
`measure.marching_cubes` looks at every cube of 8 neighbouring voxels; if some corners are above the iso-value and some below,
it places triangles inside that cube. It returns:

- `verts`: the corner points of the triangles, in mm (because we pass `spacing`), in the order **(z, y, x)**
- `faces`: the triangles, each as 3 indices into `verts`

| Surface | Iso-value | Shows |
|---|---|---|
| Lungs & airways | −500 HU | the walls of the lungs and windpipe (and the outside of the body) |
| Skin | −300 HU | the boundary between air and the body (and the scanner table behind the body) |
| Muscle & organs | 40 HU | soft tissue: the muscles around the ribs and the organs |
| Bone | 300 HU | the skeleton |
| Compact bone | 500 HU | the harder outer layer of the bones |
| Dense bone | 700 HU | only the densest bone |

Each surface also gets a color, an opacity and `visible`: whether the web page shows it when it opens.
The page downloads the other surfaces only when the user switches them on, so they do not slow down the first load.
You can add your own surfaces to the list, for example `('Fat', -100, '#f4d03f', 0.4, False)`.
Every surface adds 0.1 – 6 MB, depending on how many triangles it has.

In [ ]:
# name, iso-value (HU), color, opacity, and whether the web page shows it at the start
surface_settings = [
    ('Lungs & airways',       -500, '#7fb3d5', 0.3,  False),
    ('Skin',                  -300, '#e8a27c', 0.25, True),
    ('Muscle & organs',         40, '#c0392b', 0.4,  False),
    ('Bone',                   300, '#f2ecd8', 1.0,  True),
    ('Compact bone',           500, '#f0b429', 1.0,  False),
    ('Dense bone',             700, '#d9534f', 1.0,  False),
]

surfaces = []
for name, iso, color, opacity, visible in surface_settings:
    verts, faces, normals, values = measure.marching_cubes(small, level=iso, spacing=small_spacing)
    surfaces.append({'name': name, 'iso': iso, 'color': color, 'opacity': opacity, 'visible': visible,
                     'verts': verts, 'faces': faces})
    print(f'{name:22s} {iso:5d} HU: {len(verts):7,d} vertices, {len(faces):7,d} triangles')

## Step 6: Check a surface before exporting it

A quick matplotlib plot shows whether the iso-value picked the right tissue.
`verts` is in (z, y, x) order, so we plot column 2 (x) and column 1 (y) on the floor and column 0 (z) up.
`set_box_aspect` keeps the real proportions in mm.

**What you should see:** the spine in the middle, the ribs curving around the chest, and the collarbones and shoulder blades at the top.

In [ ]:
bone = surfaces[3]
v, f = bone['verts'], bone['faces']
xyz = v[:, [2, 1, 0]]                       # (z, y, x) -> (x, y, z)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
mesh = Poly3DCollection(xyz[f], facecolor='lightgray', edgecolor='none', alpha=1.0)
ax.add_collection3d(mesh)
ax.set_xlim(0, xyz[:, 0].max())
ax.set_ylim(0, xyz[:, 1].max())
ax.set_zlim(0, xyz[:, 2].max())
ax.set_box_aspect(np.ptp(xyz, axis=0))
ax.view_init(elev=15, azim=-60)
ax.invert_yaxis()                           # row 0 (front of the body) towards the viewer
ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)'); ax.set_zlabel('z (mm)')
ax.set_title(f"{bone['name']} surface, {bone['iso']} HU")
plt.show()

## Step 7: Export the surfaces for three.js

Each surface is saved as one small JSON file, plus a list of all surfaces (`surfaces.json`) that the page reads first:

```
data/surfaces/
├── surfaces.json      [{"name": "Bone", "iso": 300, "color": "#f2ecd8", "opacity": 1.0, "visible": true, "file": "bone_300.json"}, ...]
├── skin_-300.json     {"vertices": [x0, y0, z0, x1, y1, z1, ...], "faces": [a0, b0, c0, a1, b1, c1, ...]}
├── bone_300.json
└── ...                one file per surface
```

Three details:

1. **Axis order:** marching cubes gives (z, y, x). We reorder to **(x, y, z)**, the order people expect.
   The page then turns z (height) into three.js's *up* axis.
2. **Half-voxel shift:** marching cubes puts voxel 0 at 0 mm. In the web pages voxel 0 covers 0 … 1 voxel, with its
   **center** at half a voxel. Adding half a voxel makes the surfaces line up with the slices of index_01 – index_03.
3. **Flat lists:** three.js stores geometry as flat number arrays (`[x0, y0, z0, x1, y1, z1, …]`), so we flatten the arrays.
   Vertices are rounded to 0.1 mm to keep the files small. Face indices start at **0**, as in JavaScript
   (unlike OBJ files, which start at 1).

In [ ]:
surface_folder = os.path.join(OUT_FOLDER, 'surfaces')
os.makedirs(surface_folder, exist_ok=True)

half_voxel = np.array(small_spacing)[[2, 1, 0]] / 2      # (x, y, z) in mm
listing = []
for s in surfaces:
    xyz = s['verts'][:, [2, 1, 0]] + half_voxel             # (z, y, x) -> (x, y, z), then shift
    # 'Lungs & airways', -500  ->  'lungs_airways_-500.json'
    file_name = '_'.join(s['name'].lower().replace('&', '').split()) + f"_{s['iso']}.json"
    with open(os.path.join(surface_folder, file_name), 'w') as f:
        json.dump({'name': s['name'], 'iso': s['iso'],
                   'vertices': np.round(xyz, 1).ravel().tolist(),
                   'faces': s['faces'].ravel().tolist()},
                  f, separators=(',', ':'))
    listing.append({'name': s['name'], 'iso': s['iso'], 'color': s['color'],
                    'opacity': s['opacity'], 'visible': s['visible'], 'file': file_name})
    print(f"{file_name:32s} {os.path.getsize(os.path.join(surface_folder, file_name)) / 1e6:.1f} MB")

with open(os.path.join(surface_folder, 'surfaces.json'), 'w') as f:
    json.dump(listing, f, indent=2)

## Summary

1. We read the 68 DICOM slices, sorted them by height and converted them to **Hounsfield Units** (as in Week 11).
2. We exported a **256 × 256 × 68 volume** as a binary `int16` file for the slice and filter pages (`index_01` – `index_03`).
3. For the surfaces we **downsampled** to 128 × 128 per slice and applied a light **Gaussian filter** to remove noise.
4. **Marching cubes** turned six iso-values (from −500 to 700 HU) into triangle meshes.
5. We exported each mesh as JSON in **(x, y, z) mm**, which `index_04_iso_surfaces.html` loads into a three.js `BufferGeometry`.